# Exp10.0 — Airborne-motion ablation

Analysis-only notebook for finalized Exp10.0 artifacts. Training and probes are run by the Slurm workflow.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=Path.cwd()):
    p = start.resolve()
    for candidate in (p, *p.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'scripts').is_dir():
            return candidate
    raise RuntimeError('writingRing repo root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_10_0_airborne_motion_ablation' / 'a2_cross_user_5fold_v1'
manifest = json.loads((root / 'manifest.json').read_text())
manifest


In [ ]:
variant_summary = pd.read_csv(root / 'variant_summary.csv')
paired_summary = pd.read_csv(root / 'paired_variant_summary.csv')
variant_summary


In [ ]:
split_level = pd.read_csv(root / 'variant_split_level.csv')
pivot = split_level.pivot(index='rotation', columns='variant', values='native_test_balanced_accuracy')
ax = pivot.plot(marker='o', figsize=(9, 5))
ax.set_ylabel('Test balanced accuracy')
ax.set_xlabel('Cross-user rotation')
ax.set_title('Exp10.0 A2: D0 vs D1 vs D2')
ax.grid(True, alpha=0.25)
plt.show()


In [ ]:
paired_split = pd.read_csv(root / 'paired_variant_split_deltas.csv')
delta_col = 'native_test_balanced_accuracy_delta'
delta = paired_split.pivot(index='rotation', columns='contrast', values=delta_col)
ax = (100 * delta).plot(marker='o', figsize=(9, 5))
ax.axhline(0, linewidth=1)
ax.set_ylabel('Paired BA delta (percentage points)')
ax.set_xlabel('Cross-user rotation')
ax.set_title('Paired D0/D1/D2 effects')
ax.grid(True, alpha=0.25)
plt.show()


In [ ]:
probe_split = pd.read_csv(root / 'probe_split_level.csv')
core = [
    'input__events__fixed250_count',
    'l1__spike__fixed250_count',
    'l2__spike__fixed250_count',
]
probe_core = probe_split[probe_split.probe.isin(core)].copy()
probe_core.groupby(['variant', 'probe']).test_balanced_accuracy.agg(['mean', 'std'])


In [ ]:
probe_delta_summary = pd.read_csv(root / 'probe_paired_variant_summary.csv')
probe_delta_summary[probe_delta_summary.probe.isin(core)]
